# Task 2: Vector Database - Milvus

This notebook will showcase the usage of [Milvus](https://milvus.io) and [cuVS](https://rapids.ai/cuvs/) to run a similarity search task.

- The model used is Llama-3.2-1b for embeddings.
- The dataset is [BeIR-NQ](https://huggingface.co/datasets/BeIR/nq). The corpus consists of about 2.8 million documents coming from different information retrieval dataset. The queries are 3k questions in English.

The steps to install the latest Milvus package are available in the [Milvus documentation](https://milvus.io/docs/quickstart.md).


In [1]:
import dask.array as da
import gzip
import json
import math
import numpy as np
import os
import pickle
import pymilvus
import time
import cuvs
import cupy as cp

from cuvs import neighbors
from minio import Minio
from multiprocessing import Process
from typing import List
from tqdm import tqdm
from openai import OpenAI

from pymilvus import (
    connections, utility
)
from pymilvus.bulk_writer import LocalBulkWriter, BulkFileType

/opt/conda/envs/cuvs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setup Milvus Collection

<img src="https://milvus.io/docs/v2.5.x/assets/use-dense-vector.png" width="800" />

Milvus is an open-source Vector Database that supports cuVS for indexing on GPU.

To start adding the vectors from our dataset we will first configure Milvus itself, then create a Collection.

In [2]:
DIM = 2048
MILVUS_HOST = f"http://milvus:19530"
ID_FIELD="id"
EMBEDDING_FIELD="embedding"

collection_name = "beir_nq"

def get_milvus_client():
    return pymilvus.MilvusClient(uri=MILVUS_HOST)

def init_collection(collection_name, client):
    # Only two fields are necessary in our schema: The embeddings and their IDs.
    fields = [
        pymilvus.FieldSchema(name=ID_FIELD, dtype=pymilvus.DataType.INT64, is_primary=True),
        pymilvus.FieldSchema(name=EMBEDDING_FIELD, dtype=pymilvus.DataType.FLOAT_VECTOR, dim=DIM)
    ]

    # Create a Collection Schema, verify it.
    schema = pymilvus.CollectionSchema(fields)
    schema.verify()

    if collection_name in client.list_collections():
        print(f"Collection '{collection_name}' already exists. Deleting collection...")
        client.drop_collection(collection_name)

    # Create the Collection with our schema, and release it's resources. This will avoid Milvus automatically
    # generating an index.
    client.create_collection(collection_name, schema=schema, dimension=DIM, vector_field_name=EMBEDDING_FIELD)
    collection = pymilvus.Collection(name=collection_name, using=client._using)    # index=None
    collection.release()
    collection.drop_index()
    return collection, schema

collection, schema = init_collection(collection_name, get_milvus_client())

Collection 'beir_nq' already exists. Deleting collection...


# Creating the embeddings

In order to do a similarity search of the embeddings already present in Milvus we will have to first create the embeddings for those queries to obtain their vector representations.

The embedding model is LLama-3.2-1b.
For more information on this model you can check out it's [model card](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2?snippet_tab=Try).
For more information on how to launch NVIDIA NIM for LLMs, including how to generate an NGC API KEY, check out the [full documentation here](https://docs.nvidia.com/nim/large-language-models/latest/getting-started.html#launch-nvidia-nim-for-llms).

In [3]:
# Setup the client for the embedding model

model_endpoint = f"http://embedding:8000/v1"
model_emb_name = "nvidia/llama-3.2-nv-embedqa-1b-v2"
batch_size = 4096

def get_embeddings(
    text_data,
    endpoint:str = "https://integrate.api.nvidia.com/v1", 
    model_name: str = 'nvidia/llama-3.2-nv-embedqa-1b-v2',
    input_type: str = "passage",
    truncate: str = "NONE",
    batch_size: int = 4096,
    api_key: str = None, 
):
    api_key = api_key or os.environ.get("NVIDIA_API_KEY", None)
    client = OpenAI(
      base_url = endpoint,
      api_key = api_key
    )
    embeddings = []
    for i in range(0, len(text_data), batch_size):
      records_batch = text_data[i: i + batch_size]
      try:
          response = client.embeddings.create(
            input=records_batch,
            model=model_name,
            encoding_format="float",
            extra_body={"input_type": input_type, "truncate": truncate}  
          )
          embeddings += [entry.embedding for entry in response.data]
      except Exception as e:
          print(f"failed on {i} reason: {e}")

    return embeddings


In the following cell we will load the embeddings of the dataset that are already present on the system.
We also compute here the embeddings of the queries that will be used later in this notebook

In [4]:
# Path for intermediate file. If they are already present in the system then we can just load them.
# Otherwise we can compute them.
input_data_path = "/workspace/task/data/nq/corpus.jsonl"
text_data_path = "/workspace/task/data/records.pickle"
embeddings_path = "/workspace/task/data/embeddings.npy"
queries_data_path = "/workspace/task/data/nq/queries.jsonl"


def remove_non_ascii(text):
    return ''.join(i for i in text if ord(i)<128)

# For this notebook we will use the first 1 million vectors
n_embeddings = 1000000

if not os.path.exists(text_data_path):
    print("Creating text data")
    text_data = []
    with open(input_data_path, 'r') as fIn:
        for line in fIn:
            data = json.loads(line.strip())
            text = remove_non_ascii(data["text"]).replace('"', "'").strip()
            if len(text) == 0:
                continue  # Skip potential empty texts
            text_data.append(text)

    with open(text_data_path, 'wb') as handle:
            pickle.dump(text_data, handle)
else:
    print("Loading text data")
    text_data = pickle.load(open(text_data_path, "rb"))

text_data = text_data[:n_embeddings]

if not os.path.exists(embeddings_path):
    print("Creating embeddings")
    embeddings = get_embeddings(text, endpoint=model_endpoint, input_type="passage", truncate="END", api_key="none")
    embeddings = np.array(embeddings, dtype=np.float32)
    np.save(embeddings_path, embeddings)
else:
    print("Loading embeddings")
    embeddings = np.load(embeddings_path, mmap_mode='r')
    embeddings = embeddings[:n_embeddings]

queries = []
with open(queries_data_path, 'r') as fIn:
    for line in fIn:
        data = json.loads(line.strip())
        text = data["text"].replace('"', "'").strip()
        if len(text) == 0:
            continue  # Skip potential empty texts            
        queries.append(text)
queries_embedding = get_embeddings(queries, endpoint=model_endpoint, input_type="query", truncate="END", api_key="none")


Loading text data
Loading embeddings


# Retrieval using Milvus and NVIDIA cuVS 
Now that our embeddings are ready to be indexed, we can use Milvus and NVIDIA cuVS to do our vector search.

This is done in 3 steps: First we ingest all the vectors in the Milvus collection, then we build the Milvus index, and finally we search this Milvus index.

To ingest all the vectors in the Milvus collection we will store our vectors in MinIO, which is an object storage system.

Minio instances are launched with Milvus cluster so we can directly upload our vectors to them, and from Milvus we will be able to import that data.

Using MinIO and import data is the fastest way to ingest vectors, because you can prepare the data to be written in segments that are
of perfect size, which will bypass Milvus having to split your vectors for you.

<img src="https://milvus.io/docs/v2.5.x/assets/map-data-to-schema.png" width="1000" />

In [5]:
# Set up Minio Client
MINIO_URL = f"minio:9000"
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", None)
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", None)

# A segment is 1 GB. We want to write embeddings in files that don't exceed this size
segment_size = 1024 * 1024 * 1024

# Function to upload vectors to a MinIO bucket
def upload_to_minio(file_paths: List[List[str]], remote_paths: List[List[str]], bucket_name="a-bucket"):
    minio_client = Minio(endpoint=MINIO_URL, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
    if not minio_client.bucket_exists(bucket_name):
        minio_client.make_bucket(bucket_name)

    for local_batch, remote_batch in zip(file_paths, remote_paths):
        for local_file, remote_file in zip(local_batch, remote_batch):
            minio_client.fput_object(bucket_name, 
                                     object_name=remote_file,
                                     file_path=local_file,
                                     part_size=segment_size,
                                     num_parallel_uploads=5)

# Write the vectors in chunks of segment size, using Dask Array
def ingest_data_bulk_dask(vectors):
    if not os.path.isdir("bulk_data"):
            os.mkdir("bulk_data")
    
    n_vectors = vectors.shape[0]
    dim = vectors.shape[1]
    n_chunks = 1 + (vectors.nbytes + n_vectors * 8) // segment_size
    chunk_size = n_vectors // n_chunks
    
    da_vectors = da.from_array(vectors, chunks=(chunk_size, dim))
    da_ids = da.arange(len(vectors), chunks=(chunk_size,))
    da.to_npy_stack("bulk_data/da_embedding/", da_vectors)
    da.to_npy_stack("bulk_data/da_id/", da_ids)
    files_to_upload = []
    remote_path = []
    for chunk_nb in range(math.ceil(len(vectors) / chunk_size)):
        files_to_upload.append([f"bulk_data/da_embedding/{chunk_nb}.npy", f"bulk_data/da_id/{chunk_nb}.npy"])
        remote_path.append([f"bulk_data/da_{chunk_nb}/embedding.npy", f"bulk_data/da_{chunk_nb}/id.npy"])
    return files_to_upload, remote_path

# Loop to check import job status
def check_milvus_import_job_status(job_ids, n_vectors):
    while True:
        tasks = [utility.get_bulk_insert_state(job_id, using=get_milvus_client()._using) for job_id in job_ids]
        success = all(task.state_name == "Completed" for task in tasks)
        failure = any(task.state_name == "Failed" for task in tasks)
        for i in range(len(tasks)):
            task = tasks[i]
            if task.state_name == "Failed":
                print(task)
        if success or failure:
            break
        time.sleep(2)
    added_entities = str(sum([task.row_count for task in tasks]))
    failure = failure or added_entities != n_vectors
    if failure:
        print(f"-  Ingestion failed. Added entities: {added_entities}")
    return success and not failure

In [6]:
%%time

def ingest_data_bulk(collection_name, vectors, schema: pymilvus.CollectionSchema):
    # Step 0: Write the dataset to the disk in chunks not exceeding the segment size.
    files_to_upload, remote_path = ingest_data_bulk_dask(vectors)
    
    # Step 1: Upload dataset to a minio bucket
    upload_to_minio(files_to_upload, remote_path)
    
    # Step 2: Launch Milvus import job
    job_ids = [utility.do_bulk_insert(collection_name, batch, using=get_milvus_client()._using) for batch in remote_path]

    # Step 3: Check status of those tasks
    check_milvus_import_job_status(job_ids, str(len(vectors)))

 
ingest_data_bulk(collection_name, embeddings, schema)

CPU times: user 15.1 s, sys: 17 s, total: 32.2 s
Wall time: 7min 10s


## Creating a Milvus index on GPU

The process of vector similarity search can be greatly accelerated through the use of vector indexes, on CPU or on GPU.

In this workshop we will focus on CAGRA indexes, and learn how and when to use it.

### Using CAGRA for Hybrid GPU-CPU graph-based index

CAGRA is a graph-based nearest neighbors implementation with state-of-the art performance for both small- and large-batch sized vector searches. 

CAGRA follows the same creation steps as any other indexes in Milvus, but it is able to be adapted for querying on CPU. To do so you have to specify `adapt_for_cpu=True` as an index parameter.
This means that CAGRA is able to profit from a high training speed on GPU, as well as a low inference time on CPU, that minimize latency even on the smallest queries.

In [7]:
generic_index_params_dict = {
    "GPU_IVF_FLAT": {
        "nlist": 150     # Number of clusters
    },
    "IVF_PQ": {
        "nlist": 150,    # Number of clusters
        "m": 64,         # Vector length after quantization
        "nbits": 8       # Number of bits per vector after quantization
    },
    "GPU_IVF_PQ": {
        "nlist": 150,    # Number of clusters
        "m": 64,         # Vector length after quantization
        "nbits": 8       # Number of bits per vector after quantization
    },
    "GPU_CAGRA": {
        "intermediate_graph_degree": 128,   # Number of neighbors before pruning
        "graph_degree": 48,                 # Number of neighbors after optimization
        "adapt_for_cpu": True,              # Switch this to true for inference on CPU
        "build_algo": "NN_DESCENT"          # "NN_DESCENT" or "IVF_PQ"
    },
    "GPU_BRUTE_FORCE": {},
    "FLAT": {},
    "HNSW": {
        "M": 36,
        "efConstruction": 512
    }
}

def wait_index(collection_name):
    while True:
        progress = utility.index_building_progress(collection_name, using=get_milvus_client()._using)
        print(progress)
        if progress.get("pending_index_rows", -1) == 0:
            break
        time.sleep(5)

def create_cuvs_index(index_type, index_params=None, drop_index=True):
    
    # Drop the current index if it exists
    if collection.has_index() and drop_index:
        collection.release()
        collection.drop_index()

    # If no index params is provided, select default one
    if index_params is None:
        index_params = generic_index_params_dict[index_type]
    # Step 1: Create the index with the vectors ingested
    collection.create_index(field_name=EMBEDDING_FIELD, index_params={
        "index_type": index_type,
        "metric_type": "L2",
        "params": index_params
    })

    wait_index(collection_name=collection_name)
    # Step 2: Load the index created to make it searchable.
    collection.load()

    # Wait until the load process completes
    utility.wait_for_loading_complete(
        collection_name=collection_name,
        using=get_milvus_client()._using,
    )

In [8]:
generic_search_params_dict = {
    "GPU_IVF_FLAT": {"nprobe": 20},
    "GPU_IVF_PQ": {"nprobe": 20},
    "GPU_CAGRA": {
        #"itopk": 60, # Used if adapt_for_cpu is False
        "ef": 60      # Used if adapt_for_cpu is True
    },
    "GPU_BRUTE_FORCE": {},
    "FLAT": {},
    "HNSW": {
        "ef": 75
    }
}

def search_one(query, index_type, top_k = 5, search_params = None):
    # Generate embedding for the query using the same model
    question_embedding = get_embeddings([query], endpoint=model_endpoint, input_type="query", truncate="END", api_key="none")
    
    if search_params is None:
        search_params = generic_search_params_dict[index_type]
    hits = collection.search(
        data=np.array(question_embedding).reshape(1, DIM), anns_field=EMBEDDING_FIELD,
        param={"params": search_params}, limit=top_k
    )

    return hits

def print_search_result(query, hits):
    # Output of top-k hits
    print("Input question:", query)
    for k in range(len(hits[0])):
        print("\t{:.3f}\t{}".format(hits[0][k].distance, text_data[hits[0][k].id]))

Now we can create an index to run searches against the original corpus.

Let's create a query, the Vector Database will use our index to return the documents that are the most relevant to our question

In [9]:
%%time

index_type = "GPU_CAGRA"
create_cuvs_index(index_type)

{'total_rows': 1000000, 'indexed_rows': 0, 'pending_index_rows': 1000000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 0, 'pending_index_rows': 1000000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 0, 'pending_index_rows': 1000000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 0, 'pending_index_rows': 1000000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 0, 'pending_index_rows': 1000000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 0, 'pending_index_rows': 1000000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 125000, 'pending_index_rows': 875000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 125000, 'pending_index_rows': 875000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 125000, 'pending_index_rows': 875000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_rows': 125000, 'pending_index_rows': 875000, 'state': 'Finished'}
{'total_rows': 1000000, 'indexed_r

In [10]:
%%time

query = "Who was Grace Hopper?"
print_search_result(query, search_one(query, index_type, top_k=3))

Input question: Who was Grace Hopper?
	1.171	{'_id': 'doc671908', 'title': 'John C. Calhoun', 'text': 'In response to decades of requests, Yale President Peter Salovey announced that the university\'s Calhoun College will be renamed in 2017 to honor Grace Murray Hopper, a pioneering computer programmer, mathematician and Navy rear admiral who graduated from Yale.[148] Calhoun is commemorated elsewhere on the campus, including the exterior of Harkness Tower, a prominent campus landmark, as one of Yale\'s "Eight Worthies."[149]', 'metadata': {}}
	1.194	{'_id': 'doc371589', 'title': 'Futures techniques', 'text': 'RADM Grace Hopper, USNR was quoted as saying "Life was simple before World War II. After that, we had systems."', 'metadata': {}}
	1.223	{'_id': 'doc65433', 'title': 'History of programming languages', 'text': 'Another early programming language was devised by Grace Hopper in the US, called FLOW-MATIC. It was developed for the UNIVAC I at Remington Rand during the period from 195

In [11]:
%%time
query = "what is non controlling interest on balance sheet?"
print_search_result(query, search_one(query, index_type, top_k=3))

Input question: what is non controlling interest on balance sheet?
	1.002	{'_id': 'doc1', 'title': 'Minority interest', 'text': 'It is, however, possible (such as through special voting rights) for a controlling interest requiring consolidation to be achieved without exceeding 50% ownership, depending on the accounting standards being employed. Minority interest belongs to other investors and is reported on the consolidated balance sheet of the owning company to reflect the claim on assets belonging to other, non-controlling shareholders. Also, minority interest is reported on the consolidated income statement as a share of profit belonging to minority shareholders.', 'metadata': {}}
	1.013	{'_id': 'doc5', 'title': 'Minority interest', 'text': 'Under the International Financial Reporting Standards, the non-controlling interest is reported in accordance with IFRS 5 and is shown at the very bottom of the Equity section on the consolidated balance sheet and subsequently on the statement o

In [12]:
%%time

query = "What is creating tides?"
print_search_result(query, search_one(query, index_type, top_k=3))

Input question: What is creating tides?
	0.866	{'_id': 'doc316724', 'title': 'Water', 'text': 'Tides are the cyclic rising and falling of local sea levels caused by the tidal forces of the Moon and the Sun acting on the oceans. Tides cause changes in the depth of the marine and estuarine water bodies and produce oscillating currents known as tidal streams. The changing tide produced at a given location is the result of the changing positions of the Moon and Sun relative to the Earth coupled with the effects of Earth rotation and the local bathymetry. The strip of seashore that is submerged at high tide and exposed at low tide, the intertidal zone, is an important ecological product of ocean tides.', 'metadata': {}}
	0.891	{'_id': 'doc551541', 'title': 'Tidal power', 'text': "Tidal power is taken from the Earth's oceanic tides. Tidal forces are periodic variations in gravitational attraction exerted by celestial bodies. These forces create corresponding motions or currents in the world'

## Querying on a bigger scale

Let's search all the queries at the same time.
While doing a search with a lot of queries you can evaluate the throughput using the number of **Queries Per Second** (QPS) by dividing the number of queries per the time taken.

In [13]:
def search_many(queries_embedding, index_type, top_k = 5, search_params=None, print_throughput=False):
    if search_params is None:
        search_params = generic_search_params_dict[index_type]
    tic = time.perf_counter()
    hits = collection.search(
        data=np.array(queries_embedding), anns_field=EMBEDDING_FIELD, param={"params": search_params}, limit=top_k
    )
    toc = time.perf_counter()
    hits_id = []
    for i in range(len(hits)):
        topk_list = [hit.id for hit in hits[i]]
        # Handle cases where number of neighbors found is smaller than asked
        if len(topk_list) < top_k:
            topk_list.extend([-1] * (top_k - len(topk_list)))
        hits_id.append(topk_list)
    hits_id = np.array(hits_id)
    if print_throughput:
        print(f"Throughput: {len(queries_embedding) / (toc - tic):.1f} Queries Per Seconds")
    return hits_id

In [14]:
%%time

topk = 50
hits=search_many(queries_embedding, index_type, topk, print_throughput=True)

Throughput: 374.9 Queries Per Seconds
CPU times: user 783 ms, sys: 135 ms, total: 918 ms
Wall time: 9.23 s


## Computing recall

Now that we have seen how to do searches using approximate indexes, we can evaluate the quality of our searches by computing the recall metric.

To compute the recall metric we need to do a brute-force search to get the groundtruth, and compare it with our approximate results.

In [15]:
%%time

from cuvs.tests.ann_utils import calc_recall

def compute_groundtruth(vectors, queries, k):
    index = cuvs.neighbors.brute_force.build(vectors, metric="sqeuclidean")
    _, neighbors = cuvs.neighbors.brute_force.search(index, queries, k)
    return neighbors.copy_to_host()

gt = compute_groundtruth(cp.array(embeddings[:n_embeddings], dtype=cp.float32), cp.array(queries_embedding, dtype=cp.float32), topk)

CPU times: user 2.72 s, sys: 4.95 s, total: 7.67 s
Wall time: 7.65 s


In [16]:
calc_recall(hits, gt)

0.9817902665121668

In [17]:
import IPython 
app = IPython.Application.instance()
app.kernel.do_shutdown(restart=False)

{'status': 'ok', 'restart': False}